In [ ]:
# 1. Install Hugging Face datasets and Neo4j driver
!pip install datasets neo4j scikit-learn -q

# 2. Install PyTorch Geometric directly (No complex wheel URLs needed)
!pip install torch_geometric -q

In [ ]:
# Graph DB configuraiton
NEO4J_URI="neo4j+s://947.....databases.neo4j.io"
NEO4J_USERNAME="96...."
NEO4J_PASSWORD="gQ>...................A1I"
NEO4J_DATABASE="94b...."
AURA_INSTANCEID="49b....."
AURA_INSTANCENAME="Free instance"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from datasets import load_dataset
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


In [ ]:

# 1. Fetch UNSW-NB15 Data from Hugging Face

print(" Fetching UNSW-NB15 dataset from Hugging Face...")
dataset_stream = load_dataset("Mouwiya/UNSW-NB15", split="train[:15000]")
raw_df = dataset_stream.to_pandas()
print(f"Loaded {len(raw_df)} network flow records.")

# Synthesize source/destination IPs for topology if not explicitly separated
if 'srcip' not in raw_df.columns or 'dstip' not in raw_df.columns:
    raw_df['srcip'] = "192.168.1." + (raw_df.index % 300).astype(str)
    raw_df['dstip'] = "10.0.0." + ((raw_df.index * 13) % 300).astype(str)

# Extract binary labels (0 = Normal, 1 = Attack)
if 'label' in raw_df.columns:
    y_raw = raw_df['label'].astype(int).values
elif 'attack_cat' in raw_df.columns:
    y_raw = (raw_df['attack_cat'] != 'Normal').astype(int).values
else:
    raise ValueError("Label column not found in dataset.")

# Select numerical flow telemetry features
telemetry_features = [
    c for c in ['dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss',
                'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb',
                'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit',
                'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat']
    if c in raw_df.columns
]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(raw_df[telemetry_features].fillna(0))



In [ ]:

# 2. Build PyTorch Geometric Graph

print("Constructing Graph Structure...")

# Map unique IP nodes to continuous integer indices
unique_ips = sorted(list(set(raw_df['srcip']).union(set(raw_df['dstip']))))
ip_to_id = {ip: idx for idx, ip in enumerate(unique_ips)}

src_indices = [ip_to_id[ip] for ip in raw_df['srcip']]
dst_indices = [ip_to_id[ip] for ip in raw_df['dstip']]

edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)
edge_attr = torch.tensor(scaled_features, dtype=torch.float)
edge_labels = torch.tensor(y_raw, dtype=torch.long)

# Compute degree-based features for IP nodes
num_nodes = len(unique_ips)
in_degree = torch.zeros(num_nodes)
out_degree = torch.zeros(num_nodes)

for s, d in zip(src_indices, dst_indices):
    out_degree[s] += 1
    in_degree[d] += 1

node_features = torch.stack([in_degree, out_degree], dim=1)

# Package into PyG Data container
data = Data(
    x=node_features,
    edge_index=edge_index,
    edge_attr=edge_attr,
    edge_label=edge_labels
)

# 80/20 Train/Test Edge Split
num_edges = data.edge_index.size(1)
perm = torch.randperm(num_edges)
train_split = int(0.8 * num_edges)

train_mask = torch.zeros(num_edges, dtype=torch.bool)
test_mask = torch.zeros(num_edges, dtype=torch.bool)
train_mask[perm[:train_split]] = True
test_mask[perm[train_split:]] = True

data.train_mask = train_mask
data.test_mask = test_mask

print(f"Graph Created: {data.num_nodes} Nodes (Hosts), {data.num_edges} Edges (Flows)")



In [ ]:

# 3. Define GraphSAGE Edge Threat Classifier

class GraphSAGEThreatDetector(nn.Module):
    def __init__(self, in_node_dim, in_edge_dim, hidden_dim=64, num_classes=2):
        super(GraphSAGEThreatDetector, self).__init__()
        # Node message passing layers
        self.conv1 = SAGEConv(in_node_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        # Flow prediction MLP (Combines Source Node + Target Node + Flow Attributes)
        self.flow_classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2 + in_edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, edge_index, edge_attr):
        # 1. Propagate structural context across hosts
        h = F.relu(self.conv1(x, edge_index))
        h = self.conv2(h, edge_index)

        # 2. Extract endpoints for every edge flow
        src_nodes, dst_nodes = edge_index[0], edge_index[1]
        h_src = h[src_nodes]
        h_dst = h[dst_nodes]

        # 3. Concatenate topology representations with NetFlow features
        edge_embeddings = torch.cat([h_src, h_dst, edge_attr], dim=-1)

        # 4. Predict Normal vs Attack
        return self.flow_classifier(edge_embeddings)

# 4. Training on Colab GPU / CPU

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on Device: {device}")

model = GraphSAGEThreatDetector(
    in_node_dim=data.x.size(1),
    in_edge_dim=data.edge_attr.size(1),
    hidden_dim=64
).to(device)

data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print("\n🚀 Training GNN for Threat Detection...")
for epoch in range(1, 41):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(out[data.train_mask], data.edge_label[data.train_mask])
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            preds = out.argmax(dim=-1)
            train_acc = (preds[data.train_mask] == data.edge_label[data.train_mask]).float().mean().item()
            test_acc = (preds[data.test_mask] == data.edge_label[data.test_mask]).float().mean().item()
            print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")

In [ ]:
from neo4j import GraphDatabase

# 1. AuraDB Connection Settings
NEO4J_URI = "neo4j+s://947.....databases.neo4j.io"
NEO4J_USERNAME = "a90...."  # AuraDB default username is always 'neo4j'
NEO4J_PASSWORD = "g...."  # Must match the generated credentials.txt

# 2. Test Connection
try:
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
    )

    with driver.session() as session:
        result = session.run("RETURN 'Connected to Neo4j AuraDB successfully!' AS msg")
        print("ok", result.single()["msg"])

    driver.close()

except Exception as e:
    print(" Connection Failed:", e)

In [ ]:
from neo4j import GraphDatabase
from datasets import load_dataset

# 1. Fetch real flows from Hugging Face
print("Fetching UNSW-NB15 flow data...")
raw_df = load_dataset("Mouwiya/UNSW-NB15", split="train[:5000]").to_pandas()

# Maintain a single consistent IP generation rule (e.g. modulo 250)
if 'srcip' not in raw_df.columns:
    raw_df['srcip'] = "192.168.1." + (raw_df.index % 250).astype(str)
    raw_df['dstip'] = "10.0.0." + ((raw_df.index * 7) % 250).astype(str)

raw_df['target'] = (raw_df['attack_cat'] != 'Normal').astype(int) if 'attack_cat' in raw_df.columns else raw_df['label'].astype(int)

# 2. Ingest graph records into Neo4j AuraDB
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Optional: Clear existing graph to avoid duplicate relations
with driver.session() as session:
    print("... Clearing previous graph data...")
    session.run("MATCH (n) DETACH DELETE n")

query = """
UNWIND $batch AS flow
MERGE (src:Host {ip: flow.srcip})
MERGE (dst:Host {ip: flow.dstip})
CREATE (src)-[:COMMUNICATES_WITH {
    dur: flow.dur,
    sbytes: flow.sbytes,
    dbytes: flow.dbytes,
    is_attack: flow.is_attack
}]->(dst)
"""

batch_data = [
    {
        "srcip": row["srcip"],
        "dstip": row["dstip"],
        "dur": float(row.get("dur", 0.0)),
        "sbytes": float(row.get("sbytes", 0.0)),
        "dbytes": float(row.get("dbytes", 0.0)),
        "is_attack": int(row["target"])
    }
    for _, row in raw_df.iterrows()
]

print("⚡ Ingesting flows into Neo4j AuraDB...")
with driver.session() as session:
    batch_size = 1000
    for i in range(0, len(batch_data), batch_size):
        session.run(query, batch=batch_data[i:i+batch_size])
        print(f"Uploaded {min(i+batch_size, len(batch_data))}/{len(batch_data)} flows...")

print("ok, Ingestion complete!")
driver.close()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from neo4j import GraphDatabase
import numpy as np

# 1. Fetch Graph directly from Neo4j AuraDB
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
print(" Querying graph topology and flow attributes from Neo4j...")

cypher_query = """
MATCH (src:Host)-[r:COMMUNICATES_WITH]->(dst:Host)
RETURN src.ip AS src, dst.ip AS dst, r.dur AS dur, r.sbytes AS sbytes, r.dbytes AS dbytes, r.is_attack AS label
"""

with driver.session() as session:
    result = session.run(cypher_query)
    records = [r.data() for r in result]
driver.close()

print(f"Retrieved {len(records)} flow edges from Neo4j AuraDB.")

# 2. Build PyTorch Geometric Structure
unique_hosts = list(set([r['src'] for r in records]).union(set([r['dst'] for r in records])))
host_to_id = {host: idx for idx, host in enumerate(unique_hosts)}

src_indices = [host_to_id[r['src']] for r in records]
dst_indices = [host_to_id[r['dst']] for r in records]

edge_index = torch.tensor([src_indices, dst_indices], dtype=torch.long)

# Normalize edge features [dur, sbytes, dbytes]
features = np.array([[r['dur'], r['sbytes'], r['dbytes']] for r in records])
scaler = StandardScaler()
edge_attr = torch.tensor(scaler.fit_transform(features), dtype=torch.float)
edge_labels = torch.tensor([r['label'] for r in records], dtype=torch.long)

# Generate node degree features
num_nodes = len(unique_hosts)
in_deg = torch.zeros(num_nodes)
out_deg = torch.zeros(num_nodes)
for s, d in zip(src_indices, dst_indices):
    out_deg[s] += 1
    in_deg[d] += 1
node_features = torch.stack([in_deg, out_deg], dim=1)

# PyG Data Object
graph_data = Data(
    x=node_features,
    edge_index=edge_index,
    edge_attr=edge_attr,
    edge_label=edge_labels
)

# 80/20 Train/Test Split
num_edges = graph_data.edge_index.size(1)
perm = torch.randperm(num_edges)
train_size = int(0.8 * num_edges)

train_mask = torch.zeros(num_edges, dtype=torch.bool)
test_mask = torch.zeros(num_edges, dtype=torch.bool)
train_mask[perm[:train_size]] = True
test_mask[perm[train_size:]] = True

graph_data.train_mask = train_mask
graph_data.test_mask = test_mask

# 3. Define GraphSAGE Threat Classifier
class ThreatDetectorGNN(nn.Module):
    def __init__(self, in_node_dim, edge_feat_dim, hidden_dim=32):
        super(ThreatDetectorGNN, self).__init__()
        self.conv1 = SAGEConv(in_node_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        # Link prediction classifier
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.conv1(x, edge_index))
        h = self.conv2(h, edge_index)

        src, dst = edge_index[0], edge_index[1]
        h_src, h_dst = h[src], h[dst]

        edge_rep = torch.cat([h_src, h_dst, edge_attr], dim=-1)
        return self.edge_mlp(edge_rep)

# 4. Train Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ThreatDetectorGNN(
    in_node_dim=graph_data.x.size(1),
    edge_feat_dim=graph_data.edge_attr.size(1)
).to(device)

data = graph_data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

print("\n Training GNN for Threat Prediction...")
for epoch in range(1, 31):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index, data.edge_attr)
    loss = criterion(out[data.train_mask], data.edge_label[data.train_mask])
    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            preds = out.argmax(dim=-1)
            train_acc = (preds[data.train_mask] == data.edge_label[data.train_mask]).float().mean().item()
            test_acc = (preds[data.test_mask] == data.edge_label[data.test_mask]).float().mean().item()
            print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

#
# 1. Fetch & Inspect Nodes and Edges Data

from datasets import load_dataset
raw_df = load_dataset("Mouwiya/UNSW-NB15", split="train[:3000]").to_pandas()

if 'srcip' not in raw_df.columns:
    raw_df['srcip'] = "192.168.1." + (raw_df.index % 120).astype(str)
    raw_df['dstip'] = "10.0.0." + ((raw_df.index * 7) % 120).astype(str)

raw_df['target'] = (raw_df['attack_cat'] != 'Normal').astype(int) if 'attack_cat' in raw_df.columns else raw_df['label'].astype(int)

print("===  GRAPH DATA EXPLORATION ===")
print(f"Total Network Flows (Edges): {len(raw_df)}")
print(f"Unique Source IPs: {raw_df['srcip'].nunique()}")
print(f"Unique Dest IPs:   {raw_df['dstip'].nunique()}")
print(f"Attack vs Normal Distribution:\n{raw_df['target'].value_counts().rename({0: 'Normal', 1: 'Attack'})}\n")

# ----------------------------------------------------
# 2. Build PyG Graph & Extract Intermediate Embeddings
# ----------------------------------------------------
unique_ips = sorted(list(set(raw_df['srcip']).union(set(raw_df['dstip']))))
ip_to_id = {ip: idx for idx, ip in enumerate(unique_ips)}

src_idx = [ip_to_id[ip] for ip in raw_df['srcip']]
dst_idx = [ip_to_id[ip] for ip in raw_df['dstip']]

edge_index = torch.tensor([src_idx, dst_idx], dtype=torch.long)
scaler = StandardScaler()
features = scaler.fit_transform(raw_df[['dur', 'sbytes', 'dbytes']].fillna(0))
edge_attr = torch.tensor(features, dtype=torch.float)
edge_labels = torch.tensor(raw_df['target'].values, dtype=torch.long)

# Degree features for nodes
num_nodes = len(unique_ips)
in_deg = torch.zeros(num_nodes)
out_deg = torch.zeros(num_nodes)
for s, d in zip(src_idx, dst_idx):
    out_deg[s] += 1
    in_deg[d] += 1
x = torch.stack([in_deg, out_deg], dim=1)

# GNN with Embedding Extraction
class SAGEEmbeddingExtractor(nn.Module):
    def __init__(self, in_dim, edge_dim, hidden_dim=32):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim * 2 + edge_dim, 2)

    def forward(self, x, edge_index, edge_attr):
        # Layer 1: 1-hop neighbor aggregation
        h1 = F.relu(self.conv1(x, edge_index))
        # Layer 2: 2-hop neighbor aggregation (Node Embeddings)
        node_embeddings = self.conv2(h1, edge_index)

        # Edge Embeddings: [h_src || h_dst || raw_edge_features]
        src, dst = edge_index[0], edge_index[1]
        edge_embeddings = torch.cat([node_embeddings[src], node_embeddings[dst], edge_attr], dim=-1)

        out = self.classifier(edge_embeddings)
        return out, node_embeddings, edge_embeddings

model = SAGEEmbeddingExtractor(in_dim=2, edge_dim=3, hidden_dim=32)
out, node_emb, edge_emb = model(x, edge_index, edge_attr)

print(f"Node Embeddings Tensor Shape: {node_emb.shape} (Hosts x Latent Dims)")
print(f"Flow/Edge Embeddings Tensor Shape: {edge_emb.shape} (Flows x Combined Dims)")

# ----------------------------------------------------
# 3. Visualize Embeddings with t-SNE
# ----------------------------------------------------
tsne = TSNE(n_components=2, random_state=42)
edge_emb_2d = tsne.fit_transform(edge_emb.detach().numpy()[:1000])  # Sample first 1000 flows
sample_labels = edge_labels[:1000].numpy()

plt.figure(figsize=(8, 5))
plt.scatter(edge_emb_2d[sample_labels == 0, 0], edge_emb_2d[sample_labels == 0, 1], c='dodgerblue', label='Normal Flow', alpha=0.6)
plt.scatter(edge_emb_2d[sample_labels == 1, 0], edge_emb_2d[sample_labels == 1, 1], c='crimson', label='Attack Flow', alpha=0.6)
plt.title("2D t-SNE Projection of GNN Flow Embeddings")
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase

# 1. Connect to Neo4j AuraDB
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# 2. Fetch existing edge endpoints from Neo4j
print(" Reading existing edges directly from Neo4j...")
with driver.session() as session:
    result = session.run("""
        MATCH (s:Host)-[r:COMMUNICATES_WITH]->(d:Host)
        RETURN s.ip AS srcip, d.ip AS dstip, r.dur AS dur, r.sbytes AS sbytes, r.dbytes AS dbytes
    """)
    db_flows = [record.data() for record in result]

print(f"Retrieved {len(db_flows)} flow records from database.")

# 3. Compute GNN Predictions
model.eval()
with torch.no_grad():
    logits, _, _ = model(x, edge_index, edge_attr)
    probabilities = F.softmax(logits, dim=-1)
    predicted_classes = logits.argmax(dim=-1).numpy()
    threat_scores = probabilities[:, 1].numpy()

# 4. Build update batch aligned with database endpoints
updates = []
for i, flow in enumerate(db_flows):
    pred_idx = i % len(predicted_classes)
    updates.append({
        "srcip": flow['srcip'],
        "dstip": flow['dstip'],
        "predicted_attack": int(predicted_classes[pred_idx]),
        "threat_score": float(round(threat_scores[pred_idx], 4))
    })

# 5. Write predictions back to Neo4j relationships via Cypher
update_cypher = """
UNWIND $batch AS item
MATCH (src:Host {ip: item.srcip})-[r:COMMUNICATES_WITH]->(dst:Host {ip: item.dstip})
SET r.gnn_predicted_attack = item.predicted_attack,
    r.gnn_threat_score = item.threat_score,
    r.status = CASE WHEN item.predicted_attack = 1 THEN 'SUSPICIOUS' ELSE 'BENIGN' END
"""

print(f"⚡ Updating {len(updates)} relationships in Neo4j AuraDB...")
with driver.session() as session:
    batch_size = 1000
    for i in range(0, len(updates), batch_size):
        session.run(update_cypher, batch=updates[i:i+batch_size])
        print(f"Updated {min(i+batch_size, len(updates))}/{len(updates)} edges...")

print("ok, Update complete!")

# 6. Verify and display flagged threats
with driver.session() as session:
    res = session.run("""
        MATCH (s:Host)-[r:COMMUNICATES_WITH]->(d:Host)
        WHERE r.gnn_threat_score IS NOT NULL
        RETURN s.ip AS Source, d.ip AS Target, r.gnn_threat_score AS ThreatScore, r.status AS Status
        LIMIT 10
    """)
    print("\n🔍 Sample Flagged Threats in Neo4j:")
    for record in res:
        print(f"[{record['Status']}] {record['Source']} -> {record['Target']} | Threat Score: {record['ThreatScore']}")

driver.close()